In [13]:
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.types import interrupt, Command
from langchain_core.tools import tool

In [14]:
@tool()
def read_file(filename : str) -> str | None:
    """Reads the contents of the given filename and returns the contents"""
    return f"Contents of the file : {filename}" 

In [15]:
@tool()
def delete_file(filename : str) -> str:
    """Deletes the given file"""
    decision = interrupt(f"Shall I Delete File : {filename} ?")
        
    if decision == "yes":
        return f"Deleted file {filename}!"
    else:
        return "Sorry! Deletion cancelled by user!"

In [16]:
tools = [read_file, delete_file]

In [17]:
llm = init_chat_model('gpt-5-nano', model_provider='openai')
llm_with_tools = llm.bind_tools(tools)
# System message
sys_msg = SystemMessage(content="You are a helpful assistant. Use tools provided whenever needed.")

In [18]:
def call_llm(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [19]:
# Graph
builder = StateGraph(MessagesState)
builder.add_node("call_llm", call_llm)
builder.add_node("tools", ToolNode(tools))
 
builder.add_edge(START, "call_llm")
builder.add_conditional_edges(
    "call_llm",
     tools_condition
)
 
builder.add_edge("tools", "call_llm")
 

In [20]:
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

In [21]:
human_message = HumanMessage(content="Read file names.txt")

In [22]:
config = {"configurable": {"thread_id": "1"}}
response = graph.invoke({"messages": [human_message]}, config=config)

In [23]:
if response.get("__interrupt__"):
    prompt = response['__interrupt__'][0].value
    decision = input(prompt)
    # Resume the graph with user's input 
    response = graph.invoke(Command(resume = decision), config=config)

In [24]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Read file names.txt
================================== Ai Message ==================================
Tool Calls:
  read_file (call_IEtxgFGdsXhzlrec4qaBfeoU)
 Call ID: call_IEtxgFGdsXhzlrec4qaBfeoU
  Args:
    filename: names.txt
================================= Tool Message =================================
Name: read_file

Contents of the file : names.txt
================================== Ai Message ==================================

The content of names.txt is:
Contents of the file : names.txt

Would you like me to read another file or do something else with it?
